In [1]:
#IMPORTS
import os
import pandas as pd
import numpy as np
import joblib
import warnings


import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import ( OneHotEncoder, StandardScaler)
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    mean_squared_log_error
)

warnings.filterwarnings("ignore")

In [2]:
#MLFLOW SET UP

mlflow.set_tracking_uri("file:f:/streamlit_session/guvi_projects/smart_premium/mlruns")

mlflow.set_experiment("Insurance_Premium_Project")

print("✅ MLflow Connected")

Traceback (most recent call last):
  File "f:\streamlit_session\mlenv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "f:\streamlit_session\mlenv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "f:\streamlit_session\mlenv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1670, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "f:\streamlit_session\mlenv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1663, in _read_helper
    result = read_yaml(root, file_name)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "f:\s

✅ MLflow Connected


In [3]:
#LOAD DATA
df = pd.read_csv(
    "../data/processed/train_cleaned.csv"
)

print("✅ Data Loaded")
print(df.shape)

✅ Data Loaded
(1200000, 20)


In [4]:
#TRAIN TEST SPLIT
X = df.drop(
    columns=["Premium Amount"],
    errors="ignore"
)

y = df["Premium Amount"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
print("✅ Train Test Split Done")


✅ Train Test Split Done


In [5]:
#COLUMN TYPES
num_cols = X.select_dtypes(include="number").columns
cat_cols = X.select_dtypes(exclude="number").columns

print("Numerical Columns:", len(num_cols))
print("Categorical Columns:", len(cat_cols))

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


Numerical Columns: 10
Categorical Columns: 9


In [6]:
#MODELS
models = {

    "LinearRegression":

        LinearRegression(),
    
    "RandomForest":

        RandomForestRegressor(
            n_estimators=150,
            max_depth=15,
            min_samples_split=5,
            min_samples_leaf=2,
            max_features="sqrt",
            n_jobs=-1,
            random_state=42
        ),

    "GradientBoosting":

        GradientBoostingRegressor(
            n_estimators=150,
            learning_rate=0.08,
            max_depth=5,
            subsample=0.8,
            random_state=42
        ),


    "XGBoost":

        XGBRegressor(
            n_estimators=150,
            learning_rate=0.08,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1,
            n_jobs=-1,
            random_state=42,
            verbosity=0
        )
}


In [7]:
#TRAINING
results = []

best_model = None

best_rmsle = float("inf")

best_model_name = None


for name, model in models.items():

    print(f"\nTraining {name}...")

    # ====================================
    # Fresh Preprocessor
    # ====================================

    preprocessor = ColumnTransformer([

        (
            "num",
            num_pipeline,
            num_cols
        ),

        (
            "cat",
            cat_pipeline,
            cat_cols
        )
    ])


    # ====================================
    # Pipeline
    # ====================================

    pipe = Pipeline([

        (
            "preprocessor",
            preprocessor
        ),

        (
            "model",
            model
        )
    ])


    # ====================================
    # MLFLOW RUN
    # ====================================

    with mlflow.start_run(run_name=name):

        # ================================
        # TRAIN
        # ================================

        pipe.fit(
            X_train,
            y_train
        )

        print(f"✅ {name} Training Completed")


        # ================================
        # PREDICTION
        # ================================

        y_pred = pipe.predict(X_test)


        # ================================
        # RMSLE SAFETY
        # ================================

        y_pred_safe = np.maximum(
            y_pred,
            0
        )


        # ================================
        # METRICS
        # ================================

        rmse = np.sqrt(
            mean_squared_error(
                y_test,
                y_pred
            )
        )

        mae = mean_absolute_error(
            y_test,
            y_pred
        )

        r2 = r2_score(
            y_test,
            y_pred
        )

        rmsle = np.sqrt(
            mean_squared_log_error(
                y_test,
                y_pred_safe
            )
        )

        #STORE RESULTS
        results.append({

            "Model": name,

            "RMSE": rmse,

            "MAE": mae,

            "R2": r2,

            "RMSLE": rmsle
        })

        #SAVE MODEL IN MLFLOW
        mlflow.sklearn.log_model(
            pipe,
            name="model"
        )

        #PRINT RESULTS

        print(f"\n{name} Results")

        print(f"RMSE  : {rmse:.4f}")

        print(f"MAE   : {mae:.4f}")

        print(f"R2    : {r2:.4f}")

        print(f"RMSLE : {rmsle:.4f}")


        #BEST MODELS
        if rmsle < best_rmsle:

            best_rmsle = rmsle

            best_model = pipe

            best_model_name = name

#RESULTS DATAFRAME

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="RMSLE",
    ascending=True
)

print("\n FINAL MODEL COMPARISON\n")

print(results_df)

#CREATE MODEL FOLDER
os.makedirs("../model",exist_ok=True)

#SAVE RESULTS CSV
results_df.to_csv("../model/model_results.csv",index=False)

print("\n Results CSV Saved")

#SAVE BEST MODEL PIPELINE
joblib.dump(best_model,"../model/final_pipeline.pkl",compress=3)

print("\n Best Model Pipeline Saved")

#FINAL BEST MODEL
print("\n BEST MODEL")

print(f"Model Name : {best_model_name}")

print(f"Best RMSLE : {best_rmsle:.4f}")








Training LinearRegression...
✅ LinearRegression Training Completed


2026/05/15 11:07:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



LinearRegression Results
RMSE  : 863.2731
MAE   : 667.2841
R2    : 0.0027
RMSLE : 1.1687

Training RandomForest...
✅ RandomForest Training Completed


2026/05/15 11:07:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



RandomForest Results
RMSE  : 851.2306
MAE   : 654.9955
R2    : 0.0304
RMSLE : 1.1598

Training GradientBoosting...
✅ GradientBoosting Training Completed


2026/05/15 11:15:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



GradientBoosting Results
RMSE  : 846.1158
MAE   : 646.7214
R2    : 0.0420
RMSLE : 1.1493

Training XGBoost...
✅ XGBoost Training Completed


2026/05/15 11:15:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



XGBoost Results
RMSE  : 845.4795
MAE   : 645.0872
R2    : 0.0434
RMSLE : 1.1482

 FINAL MODEL COMPARISON

              Model        RMSE         MAE        R2     RMSLE
3           XGBoost  845.479532  645.087160  0.043424  1.148157
2  GradientBoosting  846.115801  646.721448  0.041983  1.149344
1      RandomForest  851.230630  654.995493  0.030366  1.159822
0  LinearRegression  863.273105  667.284146  0.002737  1.168678

 Results CSV Saved

 Best Model Pipeline Saved

 BEST MODEL
Model Name : XGBoost
Best RMSLE : 1.1482
